In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
# import geopandas as gpd
import pandas as pd
import seaborn as sns
from scipy.stats import qmc
from ipywidgets import IntProgress
import math as math

# importeren van ewatercycle
import ewatercycle
import ewatercycle.models
import ewatercycle.forcing

from ewatercycle_discharge import DischargeLocal

shape_file_area = 7.629080e+03 # in km^2

In [ ]:
basin_name = "boven_suriname"

# tijdsinterval
start_datum = "2019-01-01"
eind_datum = "2024-12-31"
start_datum = pd.to_datetime(start_datum, utc=True)
end_datum = pd.to_datetime(eind_datum, utc=True)
start_pd = start_datum.strftime("%Y-%m-%dT%H:%M:%SZ")
end_pd = end_datum.strftime("%Y-%m-%dT%H:%M:%SZ")
start_datum = start_pd
eind_datum = end_pd

start_calibration = start_pd
end_calibration = end_pd
start_calibration = pd.Timestamp(start_calibration).tz_localize(None)
end_calibration = pd.Timestamp(end_calibration).tz_localize(None)

print(start_datum)

# route naar shape file
shapefile = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "boven_suriname.shp"

Eigen_model = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "discharge_bmi"
Eigen_model.mkdir(exist_ok=True)

forcing_route = Path.home() / "BEP-Julian" / "BEP-Julian" / "Forcing" / "ERA5_SUR_2019_2024"/ "work" / "diagnostic" / "script" 

In [ ]:
# genereren
# ERA5_forcing = ewatercycle.forcing.sources['LumpedMakkinkForcing'].generate(
#     dataset="ERA5",
#     start_time=start_datum,
#     end_time=eind_datum,
#     shape=shapefile,
#     directory=forcing_route
# )

# inladen
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=forcing_route)

print(f"The forcing object you created: \n {ERA5_forcing}")

In [ ]:
# forcing visualiseren als controle
evap = ERA5_forcing.to_xarray()["evspsblpot"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
prec = ERA5_forcing.to_xarray()["pr"] #units: kg m^-2 s^-1 = mm/s, gesampled per dag
print(evap)
# print(prec)
fig, ax = plt.subplots(figsize=(12,5))

evap.plot(ax=ax, label='Potential evaporation')
prec.plot(ax=ax, label='Precipitation')
ax.set_ylabel('mm/s')
ax.set_xlabel("tijd")
plt.title("E en P in mm/s, sampled per dag")
ax.legend();

In [ ]:
# oppervlakte stuwmeer inladen
opp_file = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "GWSC_dams_mogwai_Brokopondo.csv"
opp = pd.read_csv(opp_file, delimiter=',', skiprows = 1)
opp = opp.rename(columns={"Time (UTC)": "Date"})
opp["Date"] = pd.to_datetime(opp["Date"], utc = True)
# opp["Date"] = opp["Date"].dt.tz_localize(None)
opp = opp[(opp["Date"] >= start_datum) &(opp["Date"] <= eind_datum)].copy()

In [ ]:
# hoogte stuwmeer inladen
height_file = Path.home() / "BEP-Julian" / "BEP-Julian" / "Suriname_Model" / "Dahiti Brokopondo water height.xlsx"
height = pd.read_excel(height_file)
height = height.iloc[:, 0].str.split(';', expand=True)
height.columns = ["datetime", "wse", "wse_u"]
height["datetime"] = pd.to_datetime(height["datetime"], utc=True)
height["wse"] = height["wse"].astype(float)
height["wse_u"] = height["wse_u"].astype(float)

In [ ]:
#Qin links van Suriname model inladen. Hierop moet discharge_bmi worden gekalibreerd
height["datetime_local_none_test"] = height["datetime"].dt.tz_localize(None)
fig, ax1 = plt.subplots(figsize=(16,5))

opp["Date"] = opp["Date"].dt.tz_localize(None)
begin_datum = pd.to_datetime(start_datum).tz_localize(None)
eind_datum = pd.to_datetime(eind_datum).tz_localize(None)
evap = evap.sel(time=slice(begin_datum, eind_datum))

height = height[
    (height["datetime_local_none_test"] >= begin_datum) &
    (height["datetime_local_none_test"] <= eind_datum)
].copy()

height = height.reset_index(drop=True)

area_interp = np.interp(height["datetime_local_none_test"].astype("int64"), opp["Date"].astype("int64"), opp["Value"])

H = height["wse"].to_numpy()
V = np.exp((H - 8)/15) - 2
Vv = V * 10**9                                                            # van km^3 naar m^3
delta_s = np.zeros(len(height["datetime_local_none_test"]))
E = np.zeros(len(height["datetime_local_none_test"]))
# opper = np.zeros(len(height["datetime_local_none_test"]))
Q_in_L = np.zeros(len(height["datetime_local_none_test"]))
E_v = np.zeros(len(height["datetime_local_none_test"]))
Q_in_L_sum = np.zeros(len(height["datetime_local_none_test"]))
Q_in_L_sum_test = np.zeros(len(height["datetime_local_none_test"]))
Q_out = 200# m^3/s 

for i in range(len(height["datetime_local_none_test"])-1):
    
    t0 = height["datetime_local_none_test"].iloc[i]
    t1 = height["datetime_local_none_test"].iloc[i+1]

    dt_seconds = (t1 - t0).total_seconds()
    delta_s[i+1] = (Vv[i+1] - Vv[i]) / dt_seconds
    
    evap_interval = evap.sel(time=slice(t0, t1))
    E[i+1] = evap_interval.mean().values * 10**-3 #/dt_seconds       # van mm/s naar m/s

    A_t = area_interp[i+1]
    E_v[i+1] = A_t * 1e6 * E[i+1]

    Q_in_L[i+1] = delta_s[i+1] + E_v[i+1] + Q_out
    Q_in_L[i+1] = np.nan_to_num(Q_in_L[i+1], nan=0.0)
    Q_in_L_sum[i] = Q_in_L.sum()
    Q_in_L_sum_test = np.cumsum(Q_in_L)

plt.plot(height["datetime_local_none_test"], Q_in_L, label = "Afvoer")
plt.plot(height["datetime_local_none_test"], E_v, label = "Neerslag")
plt.legend()
plt.title("Afvoer Surinamerivier model 1")
plt.xlabel("Datum")
plt.ylabel("Afvoer Surinamerivier [m^3/s]")
plt.grid();

In [ ]:
flow = pd.DataFrame(data=Q_in_L, index=height["datetime_local_none_test"], columns=['Q'])
flow.index = pd.to_datetime(flow.index).tz_localize(None)
flow = flow[flow.index.notna()]

## RMSE functie

In [ ]:
def RMSE(output, observed, start, end):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    
    output.index = pd.to_datetime(output.index)
    observed.index = pd.to_datetime(observed.index)

    hydro_data = pd.concat([output.reindex(observed.index, method='ffill'), observed], axis=1, keys=['model', 'observation'])
    hydro_data = hydro_data.dropna()
    
    hydro_data = hydro_data[(hydro_data.index > start) & (hydro_data.index < end)]
    
    squarediff = (hydro_data['model'] - hydro_data['observation']) ** 2
    rootMeanSquareDiff = np.sqrt(np.mean(squarediff))
    
    return rootMeanSquareDiff

## NSE functie

In [ ]:
def NSE(output, observed, start, end):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    
    output.index = pd.to_datetime(output.index)
    observed.index = pd.to_datetime(observed.index)

    hydro_data = pd.concat([output.reindex(observed.index, method='ffill'), observed], axis=1, keys=['model', 'observation'])
    hydro_data = hydro_data.dropna()
    
    hydro_data = hydro_data[(hydro_data.index > start) & (hydro_data.index < end)]

    #hier de formule van NSE invoegen
    
    top = np.sum((hydro_data['observation'] - hydro_data['model'])**2)

    bottom = np.sum((hydro_data['observation']- hydro_data['observation'].mean())**2)

    nse = 1 - (top / bottom)
    
    return nse

## log NSE functie

In [ ]:
def logNSE(output, observed, start, end):
    start = pd.to_datetime(start)
    end = pd.to_datetime(end)
    
    output.index = pd.to_datetime(output.index)
    observed.index = pd.to_datetime(observed.index)

    hydro_data = pd.concat([output.reindex(observed.index, method='ffill'), observed], axis=1, keys=['model', 'observation'])
    hydro_data = hydro_data.dropna()
    
    hydro_data = hydro_data[(hydro_data.index > start) & (hydro_data.index < end)]

    #hier de formule van NSE invoegen
    
    top = np.sum((np.log10(hydro_data['observation']) - np.log10(hydro_data['model']))**2)

    bottom = np.sum((np.log10(hydro_data['observation']) - np.log10(hydro_data['observation']).mean())**2)

    lognse = 1 - (top / bottom)
    
    return lognse

## Parameters genereren

In [ ]:
N = 1 # 50 voor show, 200 voor test, 2000 voor kalibratie

In [ ]:
param_names = ["Qgw", "alpha", "beta"]

# voor mock run
param_mins = np.array([1.29, 0.39, 1.049])
param_maxs = np.array([1.31, 0.41, 1.051])

#Fill the parameters array with N random values between each minimum and maximum 
sampler = qmc.LatinHypercube(d=len(param_names))
sample = sampler.random(n=N)
parameters = qmc.scale(sample, param_mins, param_maxs)
print(list(zip(param_names, np.round(parameters[0], decimals=3))))
# print(parameters)

In [ ]:
def mmday_to_m3s(Q_sim_mmday, frans_area):
    return (Q_sim_mmday * shape_file_area) / 86.4

## RMSE 

In [ ]:
ensemble = []

for counter in range(N): 
    # print(counter, parameters[counter])
    ensemble.append(DischargeLocal(forcing=ERA5_forcing))
    config_file, _ = ensemble[counter].setup(parameters = parameters[counter], cfg_dir = Eigen_model)
    ensemble[counter].initialize(config_file)

In [ ]:
f = IntProgress(min=0, max=N)
display(f)

objectives_RMSE = []
obs_times = height["datetime_local_none_test"]

for ensembleMember in ensemble:
    Q_m_RMSE = []
    time_RMSE = []
    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        discharge_this_timestep = ensembleMember.get_value("Q")
        Q_m_RMSE.append(discharge_this_timestep[0])
        time_RMSE.append(ensembleMember.time_as_datetime)

    Q_m_RMSE = mmday_to_m3s(np.array(Q_m_RMSE), shape_file_area)

    discharge_dataframe = pd.DataFrame({'model output': Q_m_RMSE},index=pd.to_datetime(time_RMSE))

    model_interval_Q = []

    for i in range(len(obs_times)-1):

        t0 = obs_times.iloc[i]
        t1 = obs_times.iloc[i+1]

        interval = discharge_dataframe.loc[t0:t1, "model output"]

        if len(interval) > 0:
            model_interval_Q.append(interval.mean())
        else:
            model_interval_Q.append(np.nan)

    model_interval_Q = pd.Series( model_interval_Q, index=obs_times.iloc[1:])

    observed_Q = flow["Q"].iloc[1:]

    fit_RMSE = RMSE(model_interval_Q, observed_Q, start_calibration, end_calibration)    
    objectives_RMSE.append(fit_RMSE)

    del Q_m_RMSE, time_RMSE, discharge_dataframe, fit_RMSE
    f.value += 1

for ensembleMember in ensemble:
    ensembleMember.finalize()

# kijkt naar gemiddelde tussen punten met zelfde waarde

In [ ]:
parameters_RMSE_index = np.argmin(np.array(objectives_RMSE))
if np.min(np.array(objectives_RMSE)) == np.inf:
    print("No real parameter is chosen")

parameters_RMSE = parameters[parameters_RMSE_index]

print(f'The best RMSE parameters are: {list(zip(param_names, np.round(parameters_RMSE, decimals=3)))}')
print(parameters_RMSE)
print(np.min(np.array(objectives_RMSE)))

## NSE

In [ ]:
ensemble = []

for counter in range(N): 
    ensemble.append(DischargeLocal(forcing=ERA5_forcing))
    config_file, _ = ensemble[counter].setup(parameters = parameters[counter], cfg_dir = Eigen_model)
    ensemble[counter].initialize(config_file)

In [ ]:
f = IntProgress(min=0, max=N)
display(f)

objectives_NSE = []
obs_times = height["datetime_local_none_test"]

for ensembleMember in ensemble:
    Q_m_NSE = []
    time_NSE = []
    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        discharge_this_timestep = ensembleMember.get_value("Q")
        Q_m_NSE.append(discharge_this_timestep[0])
        time_NSE.append(ensembleMember.time_as_datetime)

    Q_m_NSE = mmday_to_m3s(np.array(Q_m_NSE), shape_file_area)

    discharge_dataframe = pd.DataFrame({'model output': Q_m_NSE},index=pd.to_datetime(time_NSE))

    model_interval_Q = []

    for i in range(len(obs_times)-1):

        t0 = obs_times.iloc[i]
        t1 = obs_times.iloc[i+1]

        interval = discharge_dataframe.loc[t0:t1, "model output"]

        if len(interval) > 0:
            model_interval_Q.append(interval.mean())
        else:
            model_interval_Q.append(np.nan)

    model_interval_Q = pd.Series( model_interval_Q, index=obs_times.iloc[1:])

    observed_Q = flow["Q"].iloc[1:]

    fit_NSE = NSE(model_interval_Q, observed_Q, start_calibration, end_calibration)    
    objectives_NSE.append(fit_NSE)

    del Q_m_NSE, time_NSE, discharge_dataframe, fit_NSE
    f.value += 1
for ensembleMember in ensemble:
    ensembleMember.finalize()

# kijkt naar gemiddelde tussen punten met zelfde waarde

In [ ]:
parameters_NSE_index = np.argmax(np.array(objectives_NSE))
if np.min(np.array(objectives_NSE)) == np.inf:
    print("No real parameter is chosen")

parameters_NSE = parameters[parameters_NSE_index]

print(f'The best NSE parameters are: {list(zip(param_names, np.round(parameters_NSE, decimals=3)))}')
print(parameters_NSE)
print(np.max(np.array(objectives_NSE)))

## log NSE

In [ ]:
ensemble = []

for counter in range(N): 
    ensemble.append(DischargeLocal(forcing=ERA5_forcing))
    config_file, _ = ensemble[counter].setup(parameters = parameters[counter], cfg_dir = Eigen_model)
    ensemble[counter].initialize(config_file)

In [ ]:
f = IntProgress(min=0, max=N)
display(f)

objectives_logNSE = []
obs_times = height["datetime_local_none_test"]

for ensembleMember in ensemble:
    Q_m_logNSE = []
    time_logNSE = []
    while ensembleMember.time < ensembleMember.end_time:
        ensembleMember.update()
        discharge_this_timestep = ensembleMember.get_value("Q")
        Q_m_logNSE.append(discharge_this_timestep[0])
        time_logNSE.append(ensembleMember.time_as_datetime)

    Q_m_logNSE = mmday_to_m3s(np.array(Q_m_logNSE), shape_file_area)

    discharge_dataframe = pd.DataFrame({'model output': Q_m_logNSE},index=pd.to_datetime(time_logNSE))

    model_interval_Q = []

    for i in range(len(obs_times)-1):

        t0 = obs_times.iloc[i]
        t1 = obs_times.iloc[i+1]

        interval = discharge_dataframe.loc[t0:t1, "model output"]

        if len(interval) > 0:
            model_interval_Q.append(interval.mean())
        else:
            model_interval_Q.append(np.nan)

    model_interval_Q = pd.Series( model_interval_Q, index=obs_times.iloc[1:])

    observed_Q = flow["Q"].iloc[1:]

    fit_logNSE = logNSE(model_interval_Q, observed_Q, start_calibration, end_calibration)    
    objectives_logNSE.append(fit_logNSE)

    del Q_m_logNSE, time_logNSE, discharge_dataframe, fit_logNSE
    f.value += 1

for ensembleMember in ensemble:
    ensembleMember.finalize()

# kijkt naar gemiddelde tussen punten met zelfde waarde

In [ ]:
parameters_logNSE_index = np.argmax(np.array(objectives_logNSE))
if np.min(np.array(objectives_logNSE)) == np.inf:
    print("No real parameter is chosen")

parameters_logNSE = parameters[parameters_logNSE_index]

print(f'The best logNSE parameters are: {list(zip(param_names, np.round(parameters_logNSE, decimals=3)))}')
print(parameters_logNSE)
print(np.max(np.array(objectives_logNSE)))

## Visualiseren

run rmse

In [ ]:
# Voor test run zijn parameters overschreven met laatste kalibratie voor visualisatie
parameters_RMSE = [1.30978424, 0.39171679, 1.05045276]

model = DischargeLocal(forcing = ERA5_forcing)
config_file, _ = model.setup(parameters=parameters_RMSE, cfg_dir=Eigen_model)

model.initialize(config_file)
Q_m = []
time = []

while model.time < model.end_time:
    model.update()
    Q_m.append(model.get_value("Q")[0])
    time.append(pd.Timestamp(model.time_as_datetime))

model.finalize()

In [ ]:
model_output_mmday = pd.Series(data=Q_m, name="Modelled discharge", index=time)
model_output_m3s_RMSE = model_output_mmday * shape_file_area * 1000 / 86400

run nse

In [ ]:
# Voor test run zijn parameters overschreven met laatste kalibratie voor visualisatie
parameters_NSE = [1.30978424, 0.39171679, 1.05045276]

model = DischargeLocal(forcing = ERA5_forcing)
config_file, _ = model.setup(parameters=parameters_NSE, cfg_dir=Eigen_model)

model.initialize(config_file)
Q_m = []
time = []

while model.time < model.end_time:
    model.update()
    Q_m.append(model.get_value("Q")[0])
    time.append(pd.Timestamp(model.time_as_datetime))

model.finalize()

In [ ]:
model_output_mmday = pd.Series(data=Q_m, name="Modelled discharge", index=time)
model_output_m3s_NSE = model_output_mmday * shape_file_area * 1000 / 86400

run lognse

In [ ]:
# Voor test run zijn parameters overschreven met laatste kalibratie voor visualisatie
parameters_logNSE = [1.30978424, 0.39171679, 1.05045276]

model = DischargeLocal(forcing = ERA5_forcing)
config_file, _ = model.setup(parameters=parameters_logNSE, cfg_dir=Eigen_model)

model.initialize(config_file)
Q_m = []
time = []

while model.time < model.end_time:
    model.update()
    Q_m.append(model.get_value("Q")[0])
    time.append(pd.Timestamp(model.time_as_datetime))

model.finalize()

In [ ]:
model_output_mmday = pd.Series(data=Q_m, name="Modelled discharge", index=time)
model_output_m3s_logNSE = model_output_mmday * shape_file_area * 1000 / 86400

run test

In [ ]:
# Voor test run zijn parameters overschreven met laatste kalibratie voor visualisatie
# test run is alleen maar om handmatig aanpassingen te kunnen maken aan parameters zonder andere dingen te overschrijven. 
parameters_test = [1.30978424, 0.39171679, 1.05045276]

model = DischargeLocal(forcing = ERA5_forcing)
config_file, _ = model.setup(parameters=parameters_test, cfg_dir=Eigen_model)

model.initialize(config_file)
Q_m = []
time = []

while model.time < model.end_time:
    model.update()
    Q_m.append(model.get_value("Q")[0])
    time.append(pd.Timestamp(model.time_as_datetime))

model.finalize()

In [ ]:
model_output_mmday = pd.Series(data=Q_m, name="Modelled discharge", index=time)
model_output_m3s_test = model_output_mmday * shape_file_area * 1000 / 86400

## Grafiek

In [ ]:
# plot grafiek met discharge model 1 en model 2
plt.figure(figsize=(16, 7))

# model_output_m3s_RMSE = model_output_m3s_RMSE[(model_output_m3s_RMSE.index >= begin_datum) &(model_output_m3s_RMSE.index <= eind_datum)].copy()
# model_output_m3s_NSE = model_output_m3s_NSE[(model_output_m3s_NSE.index >= begin_datum) &(model_output_m3s_NSE.index <= eind_datum)].copy()
model_output_m3s_logNSE = model_output_m3s_logNSE[(model_output_m3s_logNSE.index >= begin_datum) &(model_output_m3s_logNSE.index <= eind_datum)].copy()
model_output_m3s_test = model_output_m3s_test[(model_output_m3s_test.index >= begin_datum) &(model_output_m3s_test.index <= eind_datum)].copy()

# model_output_m3s_logNSE.plot(label="Modelled discharge_logNSE")
# model_output_m3s_RMSE.plot(label="Modelled discharge_RMSE")
# model_output_m3s_NSE.plot(label="Modelled discharge_NSE")

model_output_m3s_test.plot(label = "Model 2")

plt.plot(height["datetime_local_none_test"], Q_in_L, label = "Model 1")

plt.xlabel("Datum")
plt.ylabel("Afvoer [m^3/s]")
plt.title("Vergelijking afvoer modellen Surinamerivier")
plt.legend()
plt.grid(True)
plt.show()
np.set_printoptions(suppress=True)

# print(f'The RMSE paramaters are: {parameters_RMSE}')
# print(f'The NSE paramters are: {parameters_NSE}')
# print(f'The logNSE paramters are: {parameters_logNSE}');

In [ ]:
dt_days = np.diff(height["datetime_local_none_test"]).astype('timedelta64[s]').astype(float) / (24 * 3600)
dt_days_full = np.zeros(len(Q_in_L))
dt_days_full[1:] = dt_days
Q_in_day_weighted = Q_in_L * dt_days_full
Q_in_day_weighted_sum = np.cumsum(Q_in_day_weighted)

In [ ]:
# plot cumulatieve discharge model 1 en model 2
plt.figure(figsize=(16, 7))

model_output_m3s_RMSE_sum = np.cumsum(model_output_m3s_RMSE)
model_output_m3s_NSE_sum = np.cumsum(model_output_m3s_NSE)
model_output_m3s_logNSE_sum = np.cumsum(model_output_m3s_logNSE)
model_output_m3s_test_sum = np.cumsum(model_output_m3s_test)

# model_output_m3s_logNSE_sum.plot(label="Modelled discharge_logNSE")
# model_output_m3s_RMSE_sum.plot(label="Modelled discharge_RMSE")
# model_output_m3s_NSE_sum.plot(label="Modelled discharge_NSE")
model_output_m3s_test_sum.plot(label="Model 2")
plt.plot(height["datetime_local_none_test"], Q_in_day_weighted_sum, label = "Model 1")

# plt.plot(height["datetime_local_none_test"], Q_in_L, label = "Q_in [m^3/s]")

plt.xlabel("Datum")
plt.ylabel("Afvoer [m^3]")
plt.title("Cumulatieve afvoer Surinamerivier van de modellen")
plt.legend()
plt.grid(True)
plt.show()

np.set_printoptions(suppress=True)

print(f'The RMSE parameters are: {parameters_RMSE}')
print(f'The NSE parameters are: {parameters_NSE}')
print(f'The logNSE parameters are: {parameters_logNSE}');

In [ ]:
def level(Qin, Qout, ERA5_forcing, A, L0):
    # Qin is array met discharge van rivier in meer, gebaseerd op E en P en dagelijks [m^3/s]
    # Qout is een standaard baseflow uit het meer [m^3/s]
    # E is verdampingsdata van ERA5, dagelijks [mm/s]
    # A is oppervlakte meer [km^2]
    # L0 is een gegeven begin hoogte [m]
    # L0 moet handmatig worden ingevoerd omdat in de toekomst niet zeker is hoe hoog het meer gaat staan op het beginpunt van de forcing data
    
    dt = 3600*24
    E = ERA5_forcing.to_xarray()["evspsblpot"] /1000 #* dt
    L = np.zeros(len(Qin))
    L[0] = L0
    A = A * 10**6
    
    for i in range(len(Qin)-1):
        dL = ((Qin.iloc[i] - Qout) / A) - E[i]
        L[i+1] = L[i] + dL*dt 
        if L[i+1] > 48.5:
            L[i+1] = 48.5
    return L

In [ ]:
# hoogte functie testen op ERA5 data
a = level(model_output_m3s_test, Q_out, ERA5_forcing, 1020, 46.5)

In [ ]:
# hoogte functie plotten voor ERA5 data
plt.figure(figsize=(16, 7))
plt.plot(model_output_m3s_logNSE.index, a, label = "Height [m]")

plt.xlabel("Date")
plt.ylabel("Height (m)")
plt.title("Height function")
plt.legend()
plt.grid(True)
plt.show()

np.set_printoptions(suppress=True)